# 2.2 환경 상태

LLM 에이전트의 상태는 텍스트, 메타데이터, 메모리의 조합이다. 이 상태가 어떻게 관찰, 저장, 검색되는지 이해하는 것이 중요하다.

### 상태의 정의
상태(state)는 에이전트가 현재 직면한 환경의 모든 관련 정보의 스냅샷이다.

### 정적 vs 동적 상태
- **정적 상태**: 시간이 지나도 변하지 않는 속성 (사용자 신원, 고정 규칙)
- **동적 상태**: 상호작용에 따라 실시간으로 변하는 정보 (대화 히스토리, 도구 상태, 메모리)

### LLM 에이전트의 상태 구성 요소
1. **입력 (Input)**: 사용자의 프롬프트 또는 환경 이벤트
2. **히스토리 (History)**: 이전의 모든 대화 턴
3. **메모리 (Memory)**: 에이전트가 학습한 패턴과 사실
4. **맥락 (Context)**: 현재 작업, 역할, 제약
5. **도구 상태 (Tool State)**: 실행된 도구의 결과 및 부작용

## LLM 에이전트의 상태 표현

### 상태 표현의 세 가지 형식

#### 1. 텍스트 기반 (Text-based)
상태 전체를 자연어로 표현한다. 가장 유연하지만 길이가 길어질 수 있다.

```
"사용자 프롬프트: 날씨가 어떻게 되나요?
이전 응답: 오늘 날씨는 맑고 따뜻합니다.
도구 호출: weather_api
도구 결과: 온도 25도, 습도 60%"
```

#### 2. 구조화 데이터 (Structured Data)
상태를 딕셔너리나 클래스로 정리하여 프로그래밍에 용이하다.

```python
{
  "user_input": "날씨가 어떻게 되나요?",
  "previous_response": "오늘 날씨는 맑고 따뜻합니다.",
  "tool_calls": ["weather_api"],
  "tool_results": {"temperature": 25, "humidity": 60}
}
```

#### 3. 메모리 기반 (Memory-based)
MemoryStream에 저장된 기억들의 집합으로 상태를 표현한다.

In [1]:
from utils_openai import *

## 실습 1: 텍스트 기반 환경 상태 구현

In [2]:
heading("텍스트 기반 환경 상태")

class EnvironmentState:
    """환경 상태를 텍스트로 표현한다"""
    def __init__(self, user_input, history=None, context="", tool_results=None):
        self.user_input = user_input
        self.history = history or []
        self.context = context
        self.tool_results = tool_results or []
    
    def to_text(self):
        """상태를 텍스트로 변환한다"""
        text = f"사용자 입력: {self.user_input}\
"
        if self.context:
            text += f"맥락: {self.context}\
"
        if self.history:
            text += f"이전 대화 ({len(self.history)} 턴):\
"
            for turn in self.history[-2:]:  # 최근 2턴만
                text += f"  - {turn}\
"
        if self.tool_results:
            text += f"도구 결과: {self.tool_results[-1]}\
"
        return text

# 상태 생성
state = EnvironmentState(
    user_input="내일 날씨가 어떨까?",
    history=["사용자: 오늘 날씨는?", "에이전트: 맑고 따뜻합니다."],
    context="날씨 조회 모드",
    tool_results=["API 호출 완료: 온도 25도"]
)

step_print(1, "환경 상태 객체 생성", str(state))
step_print(2, "텍스트 표현", state.to_text()[:80] + "...")
step_print(3, "상태 크기", f"입력: {len(state.user_input)} 글자")

print("\
✓ 텍스트 기반 상태는 자연스럽지만 길이가 길어질 수 있다.")


────────────────────────────────────────
  텍스트 기반 환경 상태
────────────────────────────────────────
  [Step 1] 환경 상태 객체 생성
  [Step 2] 텍스트 표현
    사용자 입력: 내일 날씨가 어떨까?맥락: 날씨 조회 모드이전 대화 (2 턴):  - 사용자: 오늘 날씨는?  - 에이전트: 맑고 따뜻합니다.도구...
  [Step 3] 상태 크기
    입력: 11 글자
✓ 텍스트 기반 상태는 자연스럽지만 길이가 길어질 수 있다.


## Memory Stream과 상태의 관계

### 관찰 → 기억 → 검색 → 행동

이것이 LLM 에이전트의 상태 업데이트 사이클이다:

1. **관찰 (Observation)**: 환경의 새로운 정보 (사용자 입력, 도구 결과)
2. **저장 (Storage)**: MemoryStream.add()로 기억에 추가
3. **검색 (Retrieval)**: MemoryStream.retrieve()로 관련 기억 추출
4. **상태 구성 (State Construction)**: 현재 입력 + 검색된 기억들 = 현재 상태
5. **행동 (Action)**: 상태에 기반한 응답 생성

### Episodic vs Semantic Memory
- **Episodic**: "사용자가 오늘 10시에 날씨를 물었다" (구체적 시간, 장소)
- **Semantic**: "사용자는 날씨에 관심이 많다" (추상적, 반복되는 패턴)

### 상태 공간의 크기
텍스트 기반 상태의 크기는 동적이다. 짧은 대화에서는 작지만, 긴 대화에서는 기하급수적으로 증가한다.
이를 해결하기 위해 상태 요약과 압축이 필요하다.

In [4]:
heading("MemoryStream을 활용한 동적 상태 관리")

memory = MemoryStream()

# 시간에 따른 관찰 추가
observations = [
    "사용자: 당신의 이름은?",
    "에이전트: 저는 도우미입니다.",
    "사용자: 날씨는 어떻게 되나요?",
    "에이전트: 오늘은 맑고 따뜻합니다.",
    "사용자: 감사합니다."
]

# 관찰을 순차적으로 추가
for i, obs in enumerate(observations):
    memory.add(obs)
    step_print(i+1, f"Turn {i+1}", obs[:40] + "...")

# 현재 상태 구성: 최근 입력 + 검색된 관련 기억
current_input = "내일 날씨는?"
step_print(len(observations)+1, "현재 입력", current_input)

# 관련 기억 검색
retrieved = memory.retrieve("날씨")
step_print(len(observations)+2, "검색 결과", f"{len(retrieved)} 개 관련 기억")

# 현재 상태 구성
current_state_text = f"입력: {current_input}\
검색된 기억: {len(retrieved)} 개"
step_print(len(observations)+3, "현재 상태", current_state_text)

kv_print([
    ("메모리 항목", len(memory.retrieve("전체"))),
    ("관련 항목", len(retrieved)),
    ("상태 표현", "현재 입력 + 검색된 기억")
])
print("\
✓ MemoryStream이 동적으로 상태를 구성한다.")


────────────────────────────────────────
  MemoryStream을 활용한 동적 상태 관리
────────────────────────────────────────
  [Step 1] Turn 1
    사용자: 당신의 이름은?...
  [Step 2] Turn 2
    에이전트: 저는 도우미입니다....
  [Step 3] Turn 3
    사용자: 날씨는 어떻게 되나요?...
  [Step 4] Turn 4
    에이전트: 오늘은 맑고 따뜻합니다....
  [Step 5] Turn 5
    사용자: 감사합니다....
  [Step 6] 현재 입력
    내일 날씨는?
  [Step 7] 검색 결과
    3 개 관련 기억
  [Step 8] 현재 상태
    입력: 내일 날씨는?검색된 기억: 3 개
  메모리 항목  3
  관련 항목   3
  상태 표현   현재 입력 + 검색된 기억
✓ MemoryStream이 동적으로 상태를 구성한다.


## 상태 압축과 요약

### 상태 크기 증가 문제
LLM의 맥락 윈도우는 유한하다. 매우 긴 대화에서는:
- 전체 히스토리를 포함할 수 없다
- 맥락 윈도우 내에서 상태 표현이 경쟁하게 된다
- 초기 상호작용의 중요한 정보가 손실될 수 있다

### 상태 요약 전략
1. **슬라이딩 윈도우**: 최근 N개의 턴만 유지
2. **요약 (Summarization)**: LLM으로 긴 대화를 짧은 요약으로 변환
3. **메모리 기반 압축**: 중요한 사실들만 semantic memory에 저장
4. **우선순위 지정**: 중요도가 높은 정보를 우선 유지

### 압축 비율 vs 정보 손실 트레이드오프
강하게 압축할수록 상태 크기는 작아지지만, 세부 정보가 손실된다.

In [5]:
heading("LLM을 활용한 상태 요약")

# 긴 대화 히스토리
long_history = [
    "사용자: 안녕하세요. 저는 파이썬 초보자입니다.",
    "에이전트: 환영합니다! 무엇을 배우고 싶으신가요?",
    "사용자: 반복문과 함수를 배우고 싶어요.",
    "에이전트: 좋은 선택입니다. 먼저 for 반복문부터 시작할까요?",
    "사용자: 네, 좋아요.",
    "에이전트: for 반복문은 시퀀스의 각 항목을 반복합니다."
]

# 원본 상태
original_state = "\n".join(long_history)
step_print(1, "원본 상태 길이", f"{len(original_state)} 글자")

# LLM으로 요약
summary_prompt = f"다음 대화를 한 문장으로 요약하시오:\
{original_state}"
summary = ask(summary_prompt, temperature=0.3)
step_print(2, "요약", summary)

# 압축된 상태
compressed_state = f"대화 요약: {summary}"
step_print(3, "압축된 상태 길이", f"{len(compressed_state)} 글자")

# 압축 비율 계산
compression_ratio = len(compressed_state) / len(original_state)
kv_print([
    ("원본 크기", f"{len(original_state)} 글자"),
    ("압축 크기", f"{len(compressed_state)} 글자"),
    ("압축 비율", f"{compression_ratio:.1%}")
])
print("\
✓ LLM 요약으로 상태 크기를 크게 줄일 수 있다.")


────────────────────────────────────────
  LLM을 활용한 상태 요약
────────────────────────────────────────
  [Step 1] 원본 상태 길이
    159 글자
  [Step 2] 요약
    사용자가 파이썬 초보자로서 반복문과 함수를 배우고 싶다고 하자, 에이전트가 for 반복문부터 시작하자고 제안했다.
  [Step 3] 압축된 상태 길이
    70 글자
  원본 크기  159 글자
  압축 크기  70 글자
  압축 비율  44.0%
✓ LLM 요약으로 상태 크기를 크게 줄일 수 있다.


## 실습 2: 구조화 데이터 기반 상태

In [8]:
heading("구조화된 상태 표현")

# 구조화된 상태 클래스
class StructuredState:
    def __init__(self):
        self.user_input = ""
        self.conversation_history = []
        self.current_task = ""
        self.memory_summary = ""
        self.tool_state = {}
    
    def __repr__(self):
        return f"State(task={self.current_task}, \
        history_len={len(self.conversation_history)})"

# 상태 초기화
state = StructuredState()
state.user_input = "2+2는 몇인가?"
state.current_task = "산술 문제 풀이"
state.conversation_history = [
    {"role": "user", "content": "안녕하세요"},
    {"role": "assistant", "content": "안녕하세요! 도움이 필요하신가요?"}
]
state.memory_summary = "사용자는 수학 문제를 풀고 싶어한다"
state.tool_state = {"calculator": "ready"}

step_print(1, "구조화된 상태", str(state))
step_print(2, "사용자 입력", state.user_input)
step_print(3, "현재 작업", state.current_task)
step_print(4, "메모리 요약", state.memory_summary)
step_print(5, "도구 상태", str(state.tool_state))

kv_print([
    ("구조화 여부", "Yes - 프로그래밍에 용이"),
    ("히스토리 길이", len(state.conversation_history)),
    ("작업 추적", "명확함")
])
print("\
✓ 구조화된 상태는 프로그래밍과 추론에 유리하다.")


────────────────────────────────────────
  구조화된 상태 표현
────────────────────────────────────────
  [Step 1] 구조화된 상태
    State(task=산술 문제 풀이,         history_len=2)
  [Step 2] 사용자 입력
    2+2는 몇인가?
  [Step 3] 현재 작업
    산술 문제 풀이
  [Step 4] 메모리 요약
    사용자는 수학 문제를 풀고 싶어한다
  [Step 5] 도구 상태
    {'calculator': 'ready'}
  구조화 여부   Yes - 프로그래밍에 용이
  히스토리 길이  2
  작업 추적    명확함
✓ 구조화된 상태는 프로그래밍과 추론에 유리하다.


## 핵심 정리

| 요소 | 설명 | 예시 |
|------|------|------|
| **입력** | 현재 사용자 프롬프트 | "날씨는?" |
| **히스토리** | 이전 대화 턴들 | [{"user": "안녕", "assistant": "환영합니다"}] |
| **메모리** | 학습된 패턴과 사실 | "사용자는 날씨에 관심이 많다" |
| **맥락** | 현재 작업 및 역할 | "날씨 조회 모드" |
| **도구 상태** | 외부 도구의 상태 | {"weather_api": "available"} |

### 상태 표현 방식 비교
| 방식 | 장점 | 단점 |
|------|------|------|
| **텍스트** | 유연함, 자연스러움 | 길이 증가, 구조 불명확 |
| **구조화** | 명확함, 프로그래밍 용이 | 유연성 부족 |
| **메모리** | 요약 가능, 압축됨 | 정보 손실 가능 |